In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [2]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="tarteel-ai/everyayah", 
    repo_type="dataset", local_dir="./everyayah", allow_patterns="*/*.parquet")

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 157 files: 100%|██████████| 157/157 [00:58<00:00,  2.67it/s]


'/home/ubuntu/everyayah'

In [3]:
files = glob('everyayah/*/*.parquet')
len(files)

157

In [4]:
df = pd.read_parquet(files[0])
df.head()

,audio,duration,text,reciter
0,{'bytes': b'RIFF\xe2\x82\x05\x00WAVEfmt \x10\x...,11.284875,إِنَّا أَخْلَصْنَاهُمْ بِخَالِصَةٍ ذِكْرَى الد...,alafasy
1,{'bytes': b'RIFF\xe8Q\x05\x00WAVEfmt \x10\x00\...,10.893062,وَاذْكُرْ إِسْمَاعِيلَ وَالْيَسَعَ وَذَا الْكِ...,alafasy
2,{'bytes': b'RIFF\x84\xd4\x05\x00WAVEfmt \x10\x...,11.937938,مُتَّكِئِينَ فِيهَا يَدْعُونَ فِيهَا بِفَاكِهَ...,alafasy
3,{'bytes': b'RIFFpx\x03\x00WAVEfmt \x10\x00\x00...,7.105313,هَذَا مَا تُوعَدُونَ لِيَوْمِ الْحِسَابِ,alafasy
4,{'bytes': b'RIFF\x80\x85\x03\x00WAVEfmt \x10\x...,7.209812,جَهَنَّمَ يَصْلَوْنَهَا فَبِئْسَ الْمِهَادُ,alafasy


In [5]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in files:
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in tqdm(range(len(df))):
            t = df['text'].iloc[i].strip()
            if len(t) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['audio'].iloc[i]['bytes']
            audio_np, sr = sf.read(io.BytesIO(b))
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}_{df['reciter'].iloc[i]}"
            })
        
    return data

In [6]:
data = multiprocessing(files, loop, cores = 40)

100%|██████████| 1423/1423 [02:23<00:00,  9.90it/s]


In [7]:
len(data)

234732

In [8]:
with open('everyayah.json', 'w') as fopen:
    json.dump(data, fopen)

In [9]:
audio_files = [d['audio_filename'] for d in data]

with open('everyayah-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [10]:
data[0]

{'audio_filename': 'everyayah_audio/everyayah-data-train-00039-of-00132-e0ef969055749058_0.mp3',
 'text': 'إِنَّا أَخْلَصْنَاهُمْ بِخَالِصَةٍ ذِكْرَى الدَّارِ',
 'speaker': 'everyayah_audio_alafasy'}

In [11]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

{'audio_filename': 'everyayah_audio/everyayah-data-train-00039-of-00132-e0ef969055749058_0.mp3',
 'text': 'إِنَّا أَخْلَصْنَاهُمْ بِخَالِصَةٍ ذِكْرَى الدَّارِ',
 'speaker': 'everyayah_audio_alafasy'}

In [12]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'everyayah')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00,  7.84ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  98%|█████████▊| 21.5MB / 21.9MB,   ???B/s  
Processing Files (1 / 1): 100%|██████████| 21.9MB / 21.9MB, 1.89MB/s  
Processing Files (1 / 1): 100%|██████████| 21.9MB / 21.9MB,  628kB/s  
New Data Upload: 100%|██████████| 21.9MB / 21.9MB,  628kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.22s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/63f598f06cce02b71c2443aa0b35127e0c6ea619', commit_message='Upload dataset', commit_description='', oid='63f598f06cce02b71c2443aa0b35127e0c6ea619', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [15]:
# !zip -rq everyayah_audio.zip everyayah_audio

In [16]:
# !hf upload malaysia-ai/Multilingual-TTS everyayah_audio.zip --repo-type=dataset